This is just a sample notebook used for lecture 3 laplace transforms. The code has been migrated to ```scripts/document_splitter.py```.

In [ ]:
from unstructured.partition.pdf import partition_pdf
import pytesseract
from dotenv import load_dotenv
import os
import pickle
import requests

load_dotenv()

# Load paths from .env
pytesseract.pytesseract.tesseract_cmd = os.getenv("TESSERACT_CMD")
POPPLER_PATH = os.getenv("POPPLER_PATH")

# ---------- CONFIG ----------
DOCS_DIRECTORY = "../../data/processed_lectures/lecture3_laplace_transforms"
os.makedirs(DOCS_DIRECTORY, exist_ok=True)
IMAGES_DIR = os.path.join(DOCS_DIRECTORY, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

PDF_PATH = os.path.join(DOCS_DIRECTORY, "Laplace_Transform_Properties.pdf")

In [2]:
def partition_document(file_path):
    elements = partition_pdf(
        filename=file_path,
        languages=['eng'],
        infer_table_structure=True,
        strategy='hi_res',
        extract_image_block_types=["Image", "Table"],
        extract_image_block_to_payload=False,
        extract_image_block_output_dir=IMAGES_DIR,
        poppler_path=POPPLER_PATH
    )

    print(f"✅Number of elements extracted: {len(elements)}")
    return elements

elements = partition_document(PDF_PATH)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅Number of elements extracted: 202


In [3]:
elements

In [4]:
from natsort import natsorted # to sort filenames naturally instead of lexicographically

image_files = natsorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("Extracted image files:")
for image_file in image_files:
    print(image_file)

Extracted image files:
figure-1-1.jpg
figure-1-2.jpg
figure-2-3.jpg
figure-2-4.jpg
figure-3-5.jpg
figure-3-6.jpg
figure-4-7.jpg
figure-4-8.jpg
figure-5-9.jpg
figure-5-10.jpg
figure-6-11.jpg
figure-6-12.jpg
figure-7-13.jpg
figure-7-14.jpg
figure-8-15.jpg
figure-8-16.jpg
figure-9-17.jpg
figure-9-18.jpg
figure-9-19.jpg
figure-10-20.jpg
figure-10-21.jpg
figure-10-22.jpg
figure-11-23.jpg
figure-11-24.jpg
figure-11-25.jpg
figure-12-26.jpg
figure-12-27.jpg
figure-12-28.jpg
table-9-1.jpg


In [5]:
from tqdm import tqdm
from unstructured.documents.elements import Image, Text
from google import genai
from google.genai import types

client = genai.Client()

UNIFIED_PROMPT = """
You will analyze an extracted lecture-note image and decide if it is useful.

Step 1 — Quality Check  
Mark the image as bad if it is any of the following:
- a zoomed-in crop or partial section of a larger table/figure/formula
- missing context (cut off on any edge)
- illegible, blurry, low-resolution, or faint
- decorative (logos, page numbers, headers, footers)

If the image is bad, respond with exactly:
SKIP

Step 2 — Summary  
If the image is good, produce a technically concise and accurate clear-text summary to describe the key factual information or formulas presented in the image. 

**Final Output Format:**
- Output ONLY the summary text if good
- Output ONLY "SKIP" if bad
"""

def process_images(elements, images_dir):
    # Ensure correct natural ordering (figure-1-2, figure-1-10, etc.)
    image_files = natsorted([
        f for f in os.listdir(images_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    img_i = 0
    new_elements = []

    for el in tqdm(elements, desc="Processing Images"):
        if isinstance(el, Image):
            image_path = os.path.join(images_dir, image_files[img_i])
            print(f"Processing image: {image_path}")
            img_i += 1

            # Load image bytes
            with open(image_path, "rb") as f:
                img_bytes = f.read()

            part = types.Part.from_bytes(
                data=img_bytes,
                mime_type="image/jpeg"
            )

            # ONE Gemini call → QC + summary
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[UNIFIED_PROMPT, part],
            )

            if response.text is not None:
                output = response.text.strip()

                if output == "SKIP":
                    print("Image marked as SKIP")
                    new_elements.append(Text(""))  # keep alignment
                else:
                    print(f"Summary: {output}")
                    new_elements.append(Text(output))
            else:
                new_elements.append(Text(""))  # Handle None response gracefully

        else:
            new_elements.append(el)

    return new_elements

new_elements = process_images(elements, IMAGES_DIR)

Processing Images:   0%|          | 0/202 [00:00<?, ?it/s]

Processing image: ../../data/processed_lectures/lecture3_laplace_transforms\images\figure-1-1.jpg


Processing Images:   1%|          | 2/202 [00:05<08:38,  2.59s/it]

Summary: The Laplace Transform Property 1: Linearity, states that the principle of superposition holds, meaning L[a₁f₁(t) + a₂f₂(t)] = a₁L[f₁(t)] + a₂L[f₂(t)]. However, the Laplace transform of a product of functions L(f₁(t)f₂(t)) does not equal the product of their individual Laplace transforms F₁(s)F₂(s).
Processing image: ../../data/processed_lectures/lecture3_laplace_transforms\images\figure-1-2.jpg


Processing Images:   3%|▎         | 6/202 [00:09<04:41,  1.44s/it]

...

Summary: This image presents a table summarizing Laplace Transform properties for various time-domain functions. It includes:
*   The Laplace transform pair for a specific complex exponential function: `1 + (τ3 - τ1)/(τ1 - τ2)e^(-t/τ1) + (τ3 - τ2)/(τ2 - τ1)e^(-t/τ2)` (for `τ1 ≠ τ2`) transforms to `(τ3s + 1) / (s(τ1s + 1)(τ2s + 1))`.
*   The Laplace transform of the first derivative `df/dt`: `sF(s) - f(0)`.
*   The Laplace transform of the `n`-th derivative `d^nf/dt^n`: `s^nF(s) - s^(n-1)f(0) - s^(n-2)f'(0) - ... - sf^(n-2)(0) - f^(n-1)(0)`.
*   The Laplace transform of a time-shifted function `f(t - t0)S(t - t0)`: `e^(-t0s)F(s)`.
A note specifies that `f(t)` and `F(s)` are defined for `t ≥ 0` only. The source is Seborg et al., (2011) "Process dynamics and control".
Processing image: ../../data/processed_lectures/lecture3_laplace_transforms\images\figure-12-27.jpg


Processing Images:  99%|█████████▉| 200/202 [02:35<00:06,  3.47s/it]

Summary: This table presents common Laplace transform pairs and properties. Specifically, it provides the Laplace transform for a rational function with distinct poles (Entry 23), the Laplace transform of the first derivative (Entry 24) and the n-th derivative (Entry 25) of a function f(t), and the Laplace transform for a time-shifted function f(t - t₀)S(t - t₀) (Entry 26), where f(t) and F(s) are defined for t ≥ 0.
Processing image: ../../data/processed_lectures/lecture3_laplace_transforms\images\figure-12-28.jpg


Processing Images: 100%|██████████| 202/202 [02:48<00:00,  1.20it/s]

Image marked as SKIP


In [7]:
len(new_elements)

202

In [8]:
# print unique elements
set([str(type(el)) for el in new_elements])

{"<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [9]:
# remove table elements
from unstructured.documents.elements import  Table
new_elements = [el for el in new_elements if not isinstance(el, Table)]
print(f"✅Number of elements after removing tables: {len(new_elements)}")
print(set([str(type(el)) for el in new_elements]))

✅Number of elements after removing tables: 201
{"<class 'unstructured.documents.elements.Text'>", "<class 'unstructured.documents.elements.Header'>", "<class 'unstructured.documents.elements.Title'>", "<class 'unstructured.documents.elements.NarrativeText'>"}


In [10]:
def reconstruct_text(elements):
    full_text = ""
    for el in elements:
        if hasattr(el, "text"):
            full_text += el.text + "\n\n"
    return full_text.strip()

reconstructed_text = reconstruct_text(new_elements)
print(reconstructed_text)

CH3101 - Chapter 3: Laplace Transforms

The Laplace Transform Property 1: Linearity, states that the principle of superposition holds, meaning L[a₁f₁(t) + a₂f₂(t)] = a₁L[f₁(t)] + a₂L[f₂(t)]. However, the Laplace transform of a product of functions L(f₁(t)f₂(t)) does not equal the product of their individual Laplace transforms F₁(s)F₂(s).

Property 1: Linearity

• Principle of superposition holds

but,𝐿𝐿[𝑎𝑎1𝑓𝑓1(𝑑𝑑) + 𝑎𝑎2𝑓𝑓2(𝑑𝑑)] = 𝑎𝑎1𝐿𝐿[𝑓𝑓1(𝑑𝑑)] + 𝑎𝑎2𝐿𝐿[𝑓𝑓2(𝑑𝑑)]



𝐿𝐿 𝑓𝑓1(𝑑𝑑)𝑓𝑓2(𝑑𝑑) ≠ 𝐹𝐹1(𝑠𝑠)𝐹𝐹2(𝑠𝑠)

CH3101 - Chapter 3: Laplace Transforms

The image demonstrates the linearity property of Laplace Transforms with examples for hyperbolic cosine and sine functions. It provides the definitions `cosh ax = (e^ax + e^-ax)/2` and `sinh ax = (e^ax - e^-ax)/2`. Applying linearity, it derives their Laplace Transforms as `L{cosh ax} = s / (s^2 - a^2)` and `L{sinh ax} = a / (s^2 - a^2)`.

Property 1: Linearity

𝑎𝑎𝑎𝑎

−𝑎𝑎𝑎𝑎

𝑒𝑒

+ 𝑒𝑒

cosh𝑎𝑎 𝑥𝑥 =

2

𝑎𝑎𝑎𝑎

−𝑎𝑎𝑎𝑎



𝑒𝑒

+ 𝑒𝑒

1

1

1

1

𝑠𝑠

2

2

+

=

In [13]:
print(reconstructed_text)

CH3101 - Chapter 3: Laplace Transforms

The Laplace Transform Property 1: Linearity, states that the principle of superposition holds, meaning L[a₁f₁(t) + a₂f₂(t)] = a₁L[f₁(t)] + a₂L[f₂(t)]. However, the Laplace transform of a product of functions L(f₁(t)f₂(t)) does not equal the product of their individual Laplace transforms F₁(s)F₂(s).

Property 1: Linearity

• Principle of superposition holds

but,𝐿𝐿[𝑎𝑎1𝑓𝑓1(𝑑𝑑) + 𝑎𝑎2𝑓𝑓2(𝑑𝑑)] = 𝑎𝑎1𝐿𝐿[𝑓𝑓1(𝑑𝑑)] + 𝑎𝑎2𝐿𝐿[𝑓𝑓2(𝑑𝑑)]



𝐿𝐿 𝑓𝑓1(𝑑𝑑)𝑓𝑓2(𝑑𝑑) ≠ 𝐹𝐹1(𝑠𝑠)𝐹𝐹2(𝑠𝑠)

CH3101 - Chapter 3: Laplace Transforms

The image demonstrates the linearity property of Laplace Transforms with examples for hyperbolic cosine and sine functions. It provides the definitions `cosh ax = (e^ax + e^-ax)/2` and `sinh ax = (e^ax - e^-ax)/2`. Applying linearity, it derives their Laplace Transforms as `L{cosh ax} = s / (s^2 - a^2)` and `L{sinh ax} = a / (s^2 - a^2)`.

Property 1: Linearity

𝑎𝑎𝑎𝑎

−𝑎𝑎𝑎𝑎

𝑒𝑒

+ 𝑒𝑒

cosh𝑎𝑎 𝑥𝑥 =

2

𝑎𝑎𝑎𝑎

−𝑎𝑎𝑎𝑎



𝑒𝑒

+ 𝑒𝑒

1

1

1

1

𝑠𝑠

2

2

+

=

In [17]:
from unstructured.cleaners.core import clean, replace_unicode_quotes

cleaned_text = replace_unicode_quotes(reconstructed_text)
cleaned_text = clean(cleaned_text, extra_whitespace=True, dashes=True, trailing_punctuation=True)

cleaned_text

'CH3101 Chapter 3: Laplace Transforms The Laplace Transform Property 1: Linearity, states that the principle of superposition holds, meaning L[a₁f₁(t) + a₂f₂(t)] = a₁L[f₁(t)] + a₂L[f₂(t)]. However, the Laplace transform of a product of functions L(f₁(t)f₂(t)) does not equal the product of their individual Laplace transforms F₁(s)F₂(s). Property 1: Linearity • Principle of superposition holds but,𝐿𝐿[𝑎𝑎1𝑓𝑓1(𝑑𝑑) + 𝑎𝑎2𝑓𝑓2(𝑑𝑑)] = 𝑎𝑎1𝐿𝐿[𝑓𝑓1(𝑑𝑑)] + 𝑎𝑎2𝐿𝐿[𝑓𝑓2(𝑑𝑑)] 𝐿𝐿 𝑓𝑓1(𝑑𝑑)𝑓𝑓2(𝑑𝑑) ≠ 𝐹𝐹1(𝑠𝑠)𝐹𝐹2(𝑠𝑠) CH3101 Chapter 3: Laplace Transforms The image demonstrates the linearity property of Laplace Transforms with examples for hyperbolic cosine and sine functions. It provides the definitions `cosh ax = (e^ax + e^ ax)/2` and `sinh ax = (e^ax e^ ax)/2`. Applying linearity, it derives their Laplace Transforms as `L{cosh ax} = s / (s^2 a^2)` and `L{sinh ax} = a / (s^2 a^2)`. Property 1: Linearity 𝑎𝑎𝑎𝑎 −𝑎𝑎𝑎𝑎 𝑒𝑒 + 𝑒𝑒 cosh𝑎𝑎 𝑥𝑥 = 2 𝑎𝑎𝑎𝑎 −𝑎𝑎𝑎𝑎 𝑒𝑒 + 𝑒𝑒 1 1 1 1 𝑠𝑠 2 2 + = = 𝐿𝐿 cosh𝑎𝑎 𝑥𝑥 = 𝐿𝐿 2 𝑠𝑠 − 𝑎𝑎 2 𝑠𝑠 + 𝑎𝑎

In [18]:
print(len(cleaned_text))

8646


In [28]:
import pickle
SAVE_PATH = "../../data/processed_lectures/lecture3_laplace_transforms/new_elements.pkl"

# Save
with open(SAVE_PATH, "wb") as f:
    pickle.dump(new_elements, f)
print(f"✅ new_elements saved to {SAVE_PATH}")

✅ new_elements saved to ../../data/processed_lectures/lecture3_laplace_transforms/new_elements.pkl


In [29]:
# Later, to load
with open(SAVE_PATH, "rb") as f:
    new_elements_1 = pickle.load(f)
print("✅ new_elements loaded")

new_elements_1

✅ new_elements loaded


In [31]:
# save cleaned text to txt file
with open(os.path.join(DOCS_DIRECTORY, "cleaned_text.txt"), "w", encoding="utf-8") as f:
    f.write(cleaned_text)

In [4]:
# open the saved pickle file
save_path = "../../data/processed_lectures/lecture3_laplace_transforms/new_elements_cleaned_laplace_transform.pkl"
with open(save_path, "rb") as f:
    loaded_elements = pickle.load(f)

loaded_elements

In [8]:
# count the number of header elements
from unstructured.documents.elements import Header,Footer
header_count = sum(1 for el in loaded_elements if isinstance(el, Header))
print(f"Number of Header elements: {header_count}")
footer_count = sum(1 for el in loaded_elements if isinstance(el, Footer))
print(f"Number of Footer elements: {footer_count}")

Number of Header elements: 4
Number of Footer elements: 3


In [7]:
# print out what the header elements are
for el in loaded_elements:
    if isinstance(el, Header):
        print(el)

CH3101 - Chapter 3: Laplace Transforms
CH3101 - Chapter 3: Laplace Transforms
CH3101 - Chapter 3: Laplace Transforms
CH3101 - Chapter 3: Laplace Transforms


In [9]:
# print out what the footer elements are
for el in loaded_elements:
    if isinstance(el, Footer):
        print(el)

6
8
9


In [11]:
# PRINT OUT ALL THE ELEMENTS IN THE LOADED ELEMENTS AND THEIR TYPES
for el in loaded_elements:
    print(el, type(el))

CH3101 - Chapter 3: Laplace Transforms <class 'unstructured.documents.elements.Header'>
The image presents two mathematical problems: a first-order chemical reaction described by the differential equation `dc/dt = -kc` with initial condition `c |_(t=0) = c_0`, and an algebraic equation `sc - c_0 + kc = 0` to be solved for `c`. <class 'unstructured.documents.elements.Text'>
𝑑𝑑𝑑𝑑 <class 'unstructured.documents.elements.Text'>
1 <class 'unstructured.documents.elements.Text'>
= −𝑘𝑘𝑑𝑑 <class 'unstructured.documents.elements.Text'>
 <class 'unstructured.documents.elements.Text'>
𝑑𝑑𝑑𝑑 <class 'unstructured.documents.elements.Text'>
𝑑𝑑 �𝑡𝑡=0 = 𝑑𝑑0 <class 'unstructured.documents.elements.Text'>
2 <class 'unstructured.documents.elements.Text'>
𝑠𝑠𝑑𝑑 − 𝑑𝑑0 + 𝑘𝑘𝑑𝑑 = 0 solve for 𝑑𝑑 <class 'unstructured.documents.elements.Text'>
6 <class 'unstructured.documents.elements.Footer'>
CH3101 - Chapter 3: Laplace Transforms <class 'unstructured.documents.elements.Header'>
The slide demonstrates how a first-o

In [ ]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter
